<a href="https://colab.research.google.com/github/andluizsouza/unicamp-llm-agents/blob/main/modules/01_fundamentos_ia_pln/hands_on_final_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Atividade Avaliativa**  
## **Instituto de Computação**  
## **Universidade Estadual de Campinas \- UNICAMP**  
### **Curso: INF0087** Sistemas Inteligentes com Agentes Autônomos usando Grandes Modelos de Linguagem
### **Disciplina: Fundamentos de IA Generativa e PLN**

<br>

#### **1. Descrição da Atividade:**

* Realizar o ajuste fino:
  * Usar um LLM pequeno como base, ex.: TinyLlama/TinyLlama-1.1B-Chat-v1.0, google/gemma-3-1b-pt …  
  * Para fazer o fine-tuning supervisionado, utilizar o dataset “HuggingFaceTB/smoltalk2”, subconjunto/subset “SFT” e Split “sft\_smoltalk\_multilingual\_8languages\_lang\_5\_no\_think” (link: [https://huggingface.co/datasets/HuggingFaceTB/smoltalk2/viewer/SFT/smoltalk\_multilingual\_8languages\_lang\_5\_no\_think](https://huggingface.co/datasets/HuggingFaceTB/smoltalk2/viewer/SFT/smoltalk_multilingual_8languages_lang_5_no_think)). Dica: baixe o dataset no seu gdrive (ou local) para não precisar baixá-lo a cada execução.   
  * Aplicar QLoRA para realizar o ajuste fino de instruções.  
  * Ilustrar, com exemplos, as diferenças de geração entre o modelo de base e o ajustado.

<br>

#### **2. Critérios de Avaliação:**  
Os seguintes critérios serão considerados com igual peso (25% cada):

* Complexidade:   
  * Por ser uma atividade aberta, pode ser executada em diferentes níveis de complexidade. Por exemplo: um ajuste mais refinado dos parâmetros do QLoRA, uma análise comparativa dos modelos ou a filtragem dos dados de treinamento acrescentam nota, mas não são essenciais.     
* Completude:  
  * A atividade completa inclui carregar a LLM de base, carregar o conjunto de dados para o SFT, realizar o ajuste fino com QLoRA e verificar as diferenças nas respostas. O ajuste por preferências não faz parte da avaliação (não vale nota) e é uma atividade extra para os alunos que desejarem testar suas habilidades.    
* Corretude:  
  * Ausência de bugs, código eficiente e uso adequado da linguagem Python e bibliotecas.   
* Documentação:  
  * Os códigos deverão ser acompanhados de documentação que explique cada passo no notebook, ou seja, intercalando blocos de texto e de código. A documentação mais completa e clara receberá uma nota maior.

Importante: não fazem parte da avaliação aspectos que dependem apenas do poder computacional disponível (ex.: um conjunto de treinamento muito grande não influencia a nota).

<br>

#### **3. Modo de Entrega:**

**A atividade é Individual.**
Os alunos deverão copiar este notebook, incluir documentação (blocos de texto) e códigos, abaixo deste enunciado e também, as saídas de cada bloco de código.    

A entrega do notebook (.ipynb) será feita por meio da atividade no Google ClassRoom.  


---

# Resolução da Atividade Avaliativa

## Visão Geral

Nesta atividade, realizaremos o **ajuste fino supervisionado (SFT)** do modelo **Google Gemma 3 1B** (versão pré-treinada) utilizando a técnica **QLoRA** (Quantized Low-Rank Adaptation), treinando-o com dados instrucionais em **português** extraídos do dataset SmolTalk2.

### Decisões de Arquitetura

| Aspecto | Escolha | Justificativa |
|---------|---------|---------------|
| **Modelo base** | `google/gemma-3-1b-pt` | Modelo pré-treinado (decoder-only, autorregressivo) com 1B de parâmetros. Tamanho adequado para o Colab gratuito (T4 16GB). |
| **Tokenizer** | `google/gemma-3-1b-it` | A versão instruction-tuned contém o `chat_template` necessário para formatar exemplos de SFT no formato de diálogo. |
| **Quantização** | QLoRA 4-bit (NF4) | Redução de ~8x em memória. O formato NormalFloat4 é otimizado para a distribuição de pesos de LLMs, baseado em blocos de quantização com distribuição normal. |
| **Dataset** | SmolTalk2 (SFT, multilingual) | Dataset instrucional multilíngue com 254k exemplos em 8 idiomas. |
| **Filtragem** | Idioma (português) + amostragem | Filtramos para português (reduzindo ~8x) e amostramos ~2500 exemplos para viabilizar o treino no T4. |
| **LoRA rank** | r=16 | Balanceamento entre capacidade de adaptação e eficiência. Ranks 8-32 são recomendados para modelos de 1B+ parâmetros. |

### Pipeline de Execução

```
Fase 1: Configuração → Instalar dependências, verificar GPU
Fase 2: Dados       → Carregar, filtrar (português), amostrar, analisar tokens
Fase 3: Modelo      → Carregar com quantização 4-bit, gerar baseline
Fase 4: Treino      → SFT com QLoRA via SFTTrainer
Fase 5: Avaliação   → Comparação qualitativa: modelo base vs ajustado
Fase 6: Bônus       → Ajuste por preferências (DPO)
```

---

## Fase 1: Configuração do Ambiente

### Step 1 — Instalação de Dependências e Verificação do Ambiente

Instalamos as bibliotecas essenciais para o pipeline de fine-tuning:

- **`transformers`**: Modelos e tokenizers da Hugging Face
- **`datasets`**: Carregamento e manipulação de datasets
- **`peft`**: Parameter-Efficient Fine-Tuning (LoRA/QLoRA)
- **`trl`**: Trainers especializados para LLMs (`SFTTrainer`, `DPOTrainer`)
- **`bitsandbytes`**: Quantização 4-bit para QLoRA
- **`accelerate`**: Otimização de treinamento distribuído em GPU
- **`langdetect`**: Detecção de idioma para filtragem do dataset

In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate huggingface_hub langdetect matplotlib

In [ ]:
import os
import gc
import json
import glob
import torch
import numpy as np
import matplotlib.pyplot as plt
from itertools import islice
from tqdm import tqdm

from datasets import load_dataset, load_from_disk, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

# ── Verificar GPU disponível ──
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU: {gpu_name}")
    print(f"   VRAM: {gpu_mem:.1f} GB")
    print(f"   CUDA: {torch.version.cuda}")
else:
    print("⚠️ GPU não disponível! Este notebook requer GPU para treinamento.")

print(f"   PyTorch: {torch.__version__}")

In [ ]:
# ── Login no HuggingFace (necessário para acessar o Gemma) ──
# No Google Colab, configure o token via Secrets:
#   Menu lateral → 🔑 Secrets → Adicionar "HF_TOKEN" com seu token
# Ou use login() interativo:

from huggingface_hub import login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    login(token=HF_TOKEN)
    print("✅ Login via Colab Secrets")
except Exception:
    login()
    print("✅ Login interativo concluído")

---

## Fase 2: Exploração e Preparação dos Dados

### Step 2 — Exploração e Filtragem do Dataset SmolTalk2

O dataset **[SmolTalk2](https://huggingface.co/datasets/HuggingFaceTB/smoltalk2)** contém três subconjuntos usados no pós-treinamento do SmolLM3-3B:
- **Mid**: Dados para mid-training (4.8M rows)
- **SFT**: Dados para Supervised Fine-Tuning (3.4M rows, 25 splits)
- **Preference**: Dados para alinhamento por preferências (447k rows)

Utilizaremos o split **`smoltalk_multilingual_8languages_lang_5_no_think`** do subconjunto SFT, que contém **254k exemplos** de instruções traduzidas para 8 idiomas (fr, es, it, pt, de, ar, ru, zh). O sufixo `no_think` indica que as respostas não contêm traces de raciocínio.

**Estratégia de filtragem (justificativa):**

1. **Streaming**: Carregamos progressivamente sem baixar os 254k exemplos (~vários GB)
2. **Filtro por idioma**: Usamos `langdetect` para identificar exemplos em **português**, criando um modelo mais especializado no idioma-alvo
3. **Amostragem**: Limitamos a **~2500 exemplos** — suficiente para SFT eficaz no T4, sem ultrapassar o tempo de sessão do Colab (~60 min de treino)
4. **Persistência local**: Salvamos o dataset filtrado para evitar re-download

In [ ]:
# ── Carregar dataset em modo streaming ──
ds_stream = load_dataset(
    "HuggingFaceTB/smoltalk2",
    "SFT",
    split="smoltalk_multilingual_8languages_lang_5_no_think",
    streaming=True,
)

# Inspecionar schema e primeiros exemplos
first_examples = list(islice(ds_stream, 3))

print("=== Colunas disponíveis ===")
print(list(first_examples[0].keys()))

for idx, ex in enumerate(first_examples):
    print(f"\n{'='*60}")
    print(f"Exemplo {idx + 1} — {len(ex['messages'])} turnos")
    print(f"{'='*60}")
    for msg in ex["messages"]:
        preview = msg["content"][:150]
        ellipsis = "..." if len(msg["content"]) > 150 else ""
        print(f"  [{msg['role']}]: {preview}{ellipsis}")
    if "chat_template_kwargs" in ex:
        print(f"  [chat_template_kwargs]: {ex['chat_template_kwargs']}")

In [ ]:
from langdetect import detect, DetectorFactory, LangDetectException

# Semente para reprodutibilidade na detecção de idioma
DetectorFactory.seed = 42

SAVE_PATH = "./data/smoltalk2_pt"

def detect_language(messages):
    """Detecta o idioma a partir da primeira mensagem do usuário."""
    for msg in messages:
        if msg["role"] == "user":
            try:
                return detect(msg["content"])
            except LangDetectException:
                return "unknown"
    return "unknown"


# Verificar se o dataset já foi salvo anteriormente (evitar re-download)
if os.path.exists(SAVE_PATH):
    dataset_pt = load_from_disk(SAVE_PATH)
    print(f"✅ Dataset carregado do cache local: {SAVE_PATH}")
    print(f"   Exemplos: {len(dataset_pt)}")
else:
    # Recarregar stream (streams são consumidos após iteração)
    ds_stream = load_dataset(
        "HuggingFaceTB/smoltalk2",
        "SFT",
        split="smoltalk_multilingual_8languages_lang_5_no_think",
        streaming=True,
    )

    TARGET_LANG = "pt"
    MAX_SAMPLES = 2500
    MAX_SCAN = 60000  # Limite de escaneamento para evitar timeout

    pt_examples = []
    lang_counts = {}
    total_scanned = 0

    print(f"Filtrando exemplos em português (meta: {MAX_SAMPLES})...")

    for example in tqdm(ds_stream, desc="Escaneando", total=MAX_SCAN):
        total_scanned += 1
        lang = detect_language(example["messages"])
        lang_counts[lang] = lang_counts.get(lang, 0) + 1

        if lang == TARGET_LANG:
            pt_examples.append(example)
            if len(pt_examples) >= MAX_SAMPLES:
                break

        if total_scanned >= MAX_SCAN:
            print(f"\n⚠️ Limite de escaneamento atingido ({MAX_SCAN:,} exemplos)")
            break

    # Converter para Dataset e salvar
    dataset_pt = Dataset.from_list(pt_examples)
    dataset_pt.save_to_disk(SAVE_PATH)

    print(f"\n{'='*50}")
    print(f"Total escaneado:   {total_scanned:,}")
    print(f"Exemplos em pt:    {len(pt_examples):,}")
    print(f"Dataset salvo em:  {SAVE_PATH}")
    print(f"\nDistribuição de idiomas (amostra):")
    for lang, count in sorted(lang_counts.items(), key=lambda x: -x[1])[:10]:
        pct = 100 * count / total_scanned
        print(f"  {lang}: {count:,} ({pct:.1f}%)")

# Mostrar exemplos do dataset filtrado
print(f"\n{'='*50}")
print(f"Exemplo em português:")
for msg in dataset_pt[0]["messages"]:
    preview = msg["content"][:200]
    ellipsis = "..." if len(msg["content"]) > 200 else ""
    print(f"  [{msg['role']}]: {preview}{ellipsis}")

### Step 3 — Análise de Distribuição de Tokens

Para configurar o `max_length` do treinamento, analisamos a distribuição de tokens nos exemplos filtrados. Usamos o tokenizer do Gemma 3 1B IT (que contém o `chat_template`) para tokenizar cada exemplo completo (user + assistant).

O `max_length` ideal deve cobrir a **maioria dos exemplos** sem desperdiçar memória em padding excessivo. Exemplos acima do limite serão truncados. Usamos a **mediana** ou o **percentil 90** como referência.

In [ ]:
# ── Constantes do modelo ──
MODEL_ID = "google/gemma-3-1b-pt"
TOKENIZER_ID = "google/gemma-3-1b-it"

# Carregar tokenizer (IT tem o chat_template)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)


def count_tokens_in_split(dataset_split, tokenizer):
    """Conta o número de tokens em cada exemplo após aplicar o chat_template."""
    lengths = []
    for example in tqdm(dataset_split, desc="Contando tokens"):
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
        tokens = tokenizer(text, add_special_tokens=False)["input_ids"]
        lengths.append(len(tokens))
    return np.array(lengths)


# Calcular distribuição de tokens
token_lengths = count_tokens_in_split(dataset_pt, tokenizer)

print("=== Distribuição de Tokens ===")
print(f"  Mínimo:       {token_lengths.min():,}")
print(f"  Percentil 25: {int(np.percentile(token_lengths, 25)):,}")
print(f"  Mediana:      {int(np.median(token_lengths)):,}")
print(f"  Média:        {token_lengths.mean():,.1f}")
print(f"  Percentil 90: {int(np.percentile(token_lengths, 90)):,}")
print(f"  Percentil 95: {int(np.percentile(token_lengths, 95)):,}")
print(f"  Máximo:       {token_lengths.max():,}")

In [ ]:
# ── Histograma da distribuição + definição do max_length ──
MAX_LENGTH = 512  # Definido com base na análise acima

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(token_lengths, bins=50, edgecolor="black", alpha=0.7, color="#4C72B0")
ax.axvline(np.median(token_lengths), color="red", linestyle="--",
           label=f"Mediana: {int(np.median(token_lengths))}")
ax.axvline(np.percentile(token_lengths, 90), color="orange", linestyle="--",
           label=f"P90: {int(np.percentile(token_lengths, 90))}")
ax.axvline(MAX_LENGTH, color="green", linestyle="-", linewidth=2,
           label=f"max_length: {MAX_LENGTH}")
ax.set_xlabel("Número de Tokens")
ax.set_ylabel("Frequência")
ax.set_title("Distribuição de Comprimento dos Exemplos (em tokens)")
ax.legend()
plt.tight_layout()
plt.show()

covered = (token_lengths <= MAX_LENGTH).sum() / len(token_lengths) * 100
print(f"max_length={MAX_LENGTH} cobre {covered:.1f}% dos exemplos")
print(f"Exemplos truncados: {(token_lengths > MAX_LENGTH).sum()} ({100-covered:.1f}%)")

---

## Fase 3: Carregamento do Modelo e Geração Baseline

### Step 4 — Configuração da Quantização e Carregamento do Modelo

A técnica **QLoRA** combina quantização 4-bit com adaptadores LoRA de baixo rank:

1. **Quantização NF4 (NormalFloat4)**: Os pesos do modelo são comprimidos de float32/bfloat16 para 4 bits por parâmetro. O formato NF4 é otimizado para a distribuição normal dos pesos de redes neurais — divide os valores em blocos e mapeia cada bloco para 16 níveis (4 bits), maximizando a preservação de informação.

2. **Double Quantization**: Além de quantizar os pesos, quantizamos as constantes de quantização (escalas por bloco), economizando ~0.37 bits/parâmetro adicional.

3. **Computação em bfloat16**: Apesar dos pesos em 4-bit, as operações matemáticas usam bfloat16, que oferece maior estabilidade numérica que float16 por ter maior intervalo de expoentes.

**Sobre a escolha do modelo e tokenizer:**
- Carregamos o **Gemma 3 1B PT** (pré-treinado) como base — é um modelo de geração livre que apenas completa texto
- O tokenizer vem do **Gemma 3 1B IT** (instruction-tuned) — contém o `chat_template` com tags `<start_of_turn>user` / `<start_of_turn>model` necessárias para formatar os exemplos de SFT
- Durante o fine-tuning, ensinaremos o modelo PT a responder no formato definido pelo template IT

In [ ]:
# ── Configuração de quantização 4-bit (QLoRA) ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # Quantização para 4 bits
    bnb_4bit_quant_type="nf4",               # NormalFloat4: otimizado para pesos de LLMs
    bnb_4bit_compute_dtype=torch.bfloat16,   # Computação em bfloat16 (estável)
    bnb_4bit_use_double_quant=True,          # Double quantization (economia extra)
    bnb_4bit_quant_storage=torch.bfloat16,   # Armazenamento das constantes em bfloat16
)

# ── Carregar modelo base com quantização ──
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},  # Alocar tudo na GPU 0
)

# ── Carregar tokenizer (versão IT = tem chat_template) ──
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

# Informações do modelo
total_params = sum(p.numel() for p in model.parameters())
print(f"Modelo: {MODEL_ID}")
print(f"Parâmetros totais: {total_params:,}")
print(f"VRAM utilizada: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Demonstrar o chat_template do Gemma 3 ──
demo_messages = [
    {"role": "user", "content": "Olá, como vai?"},
    {"role": "assistant", "content": "Olá! Estou bem, obrigado. Como posso ajudá-lo?"},
]
formatted = tokenizer.apply_chat_template(demo_messages, tokenize=False)
print(f"\n=== Chat Template (Gemma 3) ===")
print(formatted)

### Step 5 — Geração Baseline (Modelo Base, Pré-Ajuste)

Antes do fine-tuning, demonstramos o comportamento do modelo base. O Gemma 3 1B PT é um modelo de **geração livre** (autorregressivo, decoder-only) — ele **não segue instruções**, apenas completa texto token a token com base na probabilidade condicional:

$$P(x_t | x_1, x_2, ..., x_{t-1})$$

Ao receber um prompt formatado como chat, o modelo tentará completar o texto mas sem entender a semântica de "pergunta/resposta". O resultado tipicamente é texto desconexa ou continuação do formato sem responder à pergunta.

Definimos **5 prompts de teste** variados que serão reutilizados após o fine-tuning para comparação.

In [ ]:
# ── Prompts de teste (reutilizados na comparação pós-fine-tuning) ──
TEST_PROMPTS = [
    "Explique o que é inteligência artificial em termos simples.",
    "Quais são as principais diferenças entre Python e Java?",
    "Escreva uma receita simples de bolo de chocolate.",
    "O que é aprendizado por reforço e como ele funciona?",
    "Traduza para o inglês: 'A tecnologia está transformando a educação moderna.'",
]

# ── Pipeline de geração ──
pipe_base = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

# ── Gerar respostas baseline ──
baseline_responses = []

for i, prompt in enumerate(TEST_PROMPTS):
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

    output = pipe_base(
        formatted_prompt,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        return_full_text=False,
    )

    response = output[0]["generated_text"].strip()
    baseline_responses.append(response)

    print(f"{'='*70}")
    print(f"📝 Prompt {i+1}: {prompt}")
    print(f"{'─'*70}")
    print(f"🔴 Resposta (modelo base — geração livre):")
    print(f"   {response[:400]}")
    print()

# Salvar para comparação posterior
with open("baseline_responses.json", "w", encoding="utf-8") as f:
    json.dump({"prompts": TEST_PROMPTS, "responses": baseline_responses}, f, ensure_ascii=False, indent=2)

# Liberar pipeline (mantemos o modelo para o treinamento)
del pipe_base
gc.collect()
torch.cuda.empty_cache()

---

## Fase 4: Fine-Tuning SFT com QLoRA

O **Supervised Fine-Tuning (SFT)** é a segunda etapa no pipeline de pós-treinamento de LLMs (após o pré-treinamento e antes do alinhamento por preferências). Ele transforma um modelo de geração livre em um **assistente que segue instruções**.

O processo funciona assim:
1. **Formatação**: Cada exemplo é convertido para o formato de chat via `chat_template`
2. **Treinamento**: O modelo aprende a prever a resposta do assistente dado o prompt do usuário
3. **Loss seletiva**: A perda é calculada apenas nos tokens da resposta (não no prompt)
4. **Adapters LoRA**: Em vez de atualizar os ~1B de parâmetros, treinamos apenas adaptadores de baixo rank, resultando em ~1-5% de parâmetros treináveis

### Step 6 — Configuração LoRA

O **LoRA (Low-Rank Adaptation)** injeta matrizes treináveis de baixo rank nos pesos congelados:

$$W_{final} = W_{original} + B \times A$$

Onde $A \in \mathbb{R}^{d \times r}$ e $B \in \mathbb{R}^{r \times d}$ são os adaptadores com rank $r$. Para $r = 16$ e $d = 2048$ (Gemma 1B): apenas $2 \times 2048 \times 16 = 65.536$ parâmetros por camada — uma **redução de ~99%** em relação ao fine-tuning completo.

| Parâmetro | Valor | Justificativa |
|-----------|-------|---------------|
| `r=16` | Rank dos adaptadores | Ranks 8-32 são recomendados para modelos de ~1B. r=16 equilibra capacidade e eficiência. |
| `lora_alpha=16` | Fator de escala | `alpha/r = 1.0` → escala neutra, sem amplificação dos gradientes. |
| `lora_dropout=0.05` | Regularização | Dropout leve para evitar overfitting em ~2500 exemplos. |
| `target_modules="all-linear"` | Camadas alvo | Aplica LoRA em todas as camadas lineares para máxima adaptabilidade. |
| `modules_to_save` | `lm_head`, `embed_tokens` | Estas camadas são treinadas integralmente — necessárias para adaptar a distribuição de saída e os embeddings de entrada. |

In [ ]:
# ── Configuração LoRA ──
lora_config = LoraConfig(
    r=16,                                              # Rank dos adaptadores
    lora_alpha=16,                                     # Fator de escala (alpha/r = 1.0)
    lora_dropout=0.05,                                 # Regularização
    bias="none",                                       # Não treinar bias
    target_modules="all-linear",                       # Todas as camadas lineares
    task_type="CAUSAL_LM",                             # Modelagem de linguagem causal
    modules_to_save=["lm_head", "embed_tokens"],       # Treinamento completo nestas camadas
)

print("=== Configuração LoRA ===")
print(f"  Rank (r):          {lora_config.r}")
print(f"  Alpha:             {lora_config.lora_alpha}")
print(f"  Scaling (alpha/r): {lora_config.lora_alpha / lora_config.r}")
print(f"  Dropout:           {lora_config.lora_dropout}")
print(f"  Target modules:    {lora_config.target_modules}")
print(f"  Modules to save:   {lora_config.modules_to_save}")

### Step 7 — Configuração do Treinamento (SFTConfig)

O `SFTTrainer` (da biblioteca `trl`) gerencia o pipeline completo de fine-tuning supervisionado. As principais otimizações para o T4 (16GB VRAM):

**Otimizações de memória:**
- **`gradient_checkpointing=True`**: Recomputa ativações intermediárias durante o backward pass em vez de armazená-las em memória, economizando ~60% da VRAM ao custo de ~20% mais tempo
- **`paged_adamw_8bit`**: Otimizador AdamW com estados em 8-bit e paging automático para CPU quando necessário
- **`gradient_accumulation_steps=4`**: Simula um batch de 8 (2×4) sem consumir a memória de um batch de 8

**Decisões de treinamento:**
- **`packing=True`**: Combina exemplos curtos em uma mesma sequência de `max_length` tokens, maximizando a utilização do hardware — sem packing, sequências curtas desperdiçam compute em padding
- **`num_train_epochs=1`**: Uma única época é suficiente para ~2500 exemplos; mais épocas aumentariam o risco de overfitting
- **`learning_rate=2e-4`**: Taxa padrão para LoRA, conservadora para evitar catástrofe do esquecimento (*catastrophic forgetting*)
- **`bf16=True`**: Mixed precision em bfloat16, mais estável que float16 para LLMs por ter maior intervalo de expoentes

In [ ]:
os.environ["WANDB_DISABLED"] = "true"  # Desabilitar Weights & Biases

OUTPUT_DIR = "./outputs-sft"

# ── Configuração de treinamento ──
training_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    # Comprimento e packing
    max_length=MAX_LENGTH,                  # Baseado na análise de tokens (Step 3)
    packing=True,                           # Empacotar exemplos curtos

    # Épocas e batch
    num_train_epochs=1,                     # 1 época (amostra de ~2500 exemplos)
    per_device_train_batch_size=2,          # Batch por GPU
    gradient_accumulation_steps=4,          # Batch efetivo: 2 × 4 = 8

    # Otimizações de memória
    gradient_checkpointing=True,            # Recomputar ativações
    optim="paged_adamw_8bit",               # Otimizador eficiente

    # Learning rate
    learning_rate=2e-4,                     # Taxa padrão para LoRA
    warmup_steps=100,                       # Aquecimento gradual
    lr_scheduler_type="constant",           # Constante após warmup

    # Precisão
    fp16=False,                             # Instável para LLMs
    bf16=True,                              # bfloat16 (estável)

    # Regularização
    max_grad_norm=0.3,                      # Gradient clipping

    # Logging e checkpoints
    logging_steps=25,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,                     # Manter 2 checkpoints recentes
    save_only_model=True,

    # Tokenização
    dataset_kwargs={
        "add_special_tokens": False,        # Template já inclui tokens especiais
        "append_concat_token": True,        # Separação no packing
    },
)

# ── Instanciar SFTTrainer ──
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_pt,
    args=training_config,
    peft_config=lora_config,
    processing_class=tokenizer,
)

# Exibir parâmetros treináveis
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("=== Parâmetros do Modelo com LoRA ===")
print(f"  Total:      {total_params:,}")
print(f"  Treináveis: {trainable_params:,}")
print(f"  Percentual: {100 * trainable_params / total_params:.2f}%")
print(f"  Redução:    {100 * (1 - trainable_params / total_params):.2f}%")
print(f"\n  VRAM após setup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

### Step 8 — Executar Treinamento

O `SFTTrainer` cuida de todo o pipeline:
1. Formatar cada exemplo usando o `chat_template` do tokenizer
2. Tokenizar e criar batches com packing de exemplos curtos
3. Computar a loss apenas na resposta do assistente
4. Atualizar apenas os adaptadores LoRA (pesos base congelados em 4-bit)
5. Salvar checkpoints periódicos

> **Tempo estimado**: ~30-60 minutos no Google Colab T4.

In [ ]:
# ── Executar treinamento SFT com QLoRA ──
print("Iniciando treinamento SFT com QLoRA...")
print(f"  Dataset:       {len(dataset_pt)} exemplos em português")
print(f"  Batch efetivo: {training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps}")
print(f"  max_length:    {MAX_LENGTH}")
print(f"  Épocas:        {int(training_config.num_train_epochs)}")
print()

trainer.train()

print(f"\n✅ Treinamento concluído!")
print(f"   VRAM final: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Salvar modelo e tokenizer ──
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Listar checkpoints
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
print(f"Checkpoints salvos: {len(checkpoints)}")
for cp in checkpoints:
    print(f"  {cp}")
print(f"\nModelo final salvo em: {OUTPUT_DIR}")

# ── Liberar memória do treinamento ──
del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM após limpeza: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---

## Fase 5: Avaliação e Comparação

### Step 9 — Carregar Modelo Ajustado para Inferência

> ⚠️ **Recomendação**: No Google Colab, reinicie o runtime antes desta seção para liberar a VRAM do treinamento: `Runtime → Restart runtime`. Em seguida, execute a célula abaixo (que re-importa as dependências e recarrega o modelo).

Carregamos o modelo base com quantização 4-bit e aplicamos os adaptadores LoRA salvos no checkpoint. Usamos `PeftModel.from_pretrained()` para carregar os pesos LoRA sobre o modelo base congelado.

Para comparação, utilizamos o método `model.disable_adapter()` que desabilita temporariamente os adaptadores LoRA, permitindo gerar com o modelo base puro sem precisar carregar uma segunda cópia.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# EXECUTAR APÓS REINICIAR O RUNTIME (se reiniciou)
# ═══════════════════════════════════════════════════════════════════════

import os, gc, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from peft import PeftModel

# Login HuggingFace
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()

# Constantes
MODEL_ID = "google/gemma-3-1b-pt"
TOKENIZER_ID = "google/gemma-3-1b-it"
OUTPUT_DIR = "./outputs-sft"

# Quantização (mesma configuração do treinamento)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# Carregar modelo base + adaptadores LoRA
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)

model_ft = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model_ft.config.use_cache = True
model_ft.gradient_checkpointing_disable()

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

print(f"✅ Modelo ajustado carregado de: {OUTPUT_DIR}")
print(f"   VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

### Step 10 — Comparação: Modelo Base vs Modelo Ajustado

Comparamos respostas dos dois modelos nos **mesmos 5 prompts** definidos no Step 5, utilizando o método `model.disable_adapter()` para alternar entre modelo base e ajustado sem duplicar o modelo em memória.

**Esperamos que o modelo ajustado:**
1. Responda diretamente à pergunta (em vez de apenas completar texto)
2. Use o formato de chat estruturado (turnos user/model)
3. Gere respostas predominantemente em português (idioma dominante no treino)
4. Produza respostas mais coerentes e úteis

Testamos também prompts **fora do domínio** para avaliar a generalização.

In [ ]:
# ── Prompts de teste ──
TEST_PROMPTS = [
    "Explique o que é inteligência artificial em termos simples.",
    "Quais são as principais diferenças entre Python e Java?",
    "Escreva uma receita simples de bolo de chocolate.",
    "O que é aprendizado por reforço e como ele funciona?",
    "Traduza para o inglês: 'A tecnologia está transformando a educação moderna.'",
]

# Prompts fora do domínio (para avaliar generalização)
OOD_PROMPTS = [
    "Write a Python function that calculates the Fibonacci sequence.",  # Inglês
    "Quem foi Alan Turing e qual sua contribuição para a computação?",  # Conhecimento geral
]

ALL_PROMPTS = TEST_PROMPTS + OOD_PROMPTS

# ── Pipeline de geração ──
pipe_ft = pipeline(
    "text-generation",
    model=model_ft,
    tokenizer=tokenizer,
)


def generate_response(pipe, prompt, max_new_tokens=300, use_adapter=True):
    """Gera resposta com ou sem adaptadores LoRA."""
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

    stop_token_ids = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<end_of_turn>"),
    ]

    if use_adapter:
        output = pipe(
            formatted, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.3, top_p=0.9,
            return_full_text=False, eos_token_id=stop_token_ids,
        )
    else:
        with model_ft.disable_adapter():
            output = pipe(
                formatted, max_new_tokens=max_new_tokens,
                do_sample=True, temperature=0.7, top_p=0.9,
                return_full_text=False, eos_token_id=stop_token_ids,
            )

    return output[0]["generated_text"].strip()


# ── Gerar e comparar ──
print("=" * 80)
print("  COMPARAÇÃO: MODELO BASE vs MODELO AJUSTADO (SFT + QLoRA)")
print("=" * 80)

for i, prompt in enumerate(ALL_PROMPTS):
    is_ood = i >= len(TEST_PROMPTS)
    tag = " [FORA DO DOMÍNIO]" if is_ood else ""

    print(f"\n{'━' * 80}")
    print(f"📝 PROMPT {i+1}{tag}: {prompt}")
    print(f"{'━' * 80}")

    # Modelo base (sem adapters)
    response_base = generate_response(pipe_ft, prompt, use_adapter=False)
    print(f"\n🔴 Modelo Base (gemma-3-1b-pt):")
    print(f"   {response_base[:500]}")

    # Modelo ajustado (com adapters)
    response_ft = generate_response(pipe_ft, prompt, use_adapter=True)
    print(f"\n🟢 Modelo Ajustado (SFT + QLoRA):")
    print(f"   {response_ft[:500]}")

    print()

### Step 11 — Upload do Modelo ao HuggingFace Hub

Publicamos os adaptadores LoRA treinados no HuggingFace Hub para compartilhamento e reutilização. Apenas os adaptadores são enviados (~50-100MB), não o modelo base completo (~2GB).

In [ ]:
# ── Upload ao HuggingFace Hub ──
# Altere o nome do repositório para seu usuário
HF_REPO = "seu-usuario/gemma-3-1b-pt-sft-portuguese"  # ← ALTERAR AQUI

model_ft.push_to_hub(HF_REPO, private=False)
tokenizer.push_to_hub(HF_REPO, private=False)

print(f"✅ Modelo publicado em: https://huggingface.co/{HF_REPO}")

---

## Fase 6 (Bônus): Ajuste por Preferências — DPO

> ⚠️ **Esta seção é extra** e não faz parte da avaliação. É uma demonstração adicional para testar habilidades de alinhamento por preferências.

O **DPO (Direct Preference Optimization)** é uma alternativa simplificada ao RLHF para alinhar modelos com preferências humanas. Enquanto o RLHF requer um modelo de recompensa separado + PPO, o DPO otimiza diretamente a política do modelo usando pares de respostas preferidas/rejeitadas.

**Função de perda DPO:**

$$\mathcal{L}_{DPO}(\pi_\theta; \pi_{ref}) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)} \right) \right]$$

Onde:
- $\pi_\theta$: modelo que estamos treinando (política)
- $\pi_{ref}$: modelo de referência congelado (cópia do SFT)
- $y_w$: resposta preferida (*chosen*)
- $y_l$: resposta rejeitada (*rejected*)
- $\beta$: controla o desvio máximo em relação à referência

| Aspecto | RLHF (PPO) | DPO |
|---------|------------|-----|
| Modelos necessários | 2+ (reward + LLM) | 1 (auto-referenciado) |
| Estabilidade | Complexo, instável | Estável, direto |
| Custo computacional | Alto (múltiplas fases) | Menor (fase única) |

### Step 12 — Preparar Dataset de Preferências e Treinar DPO

> ⚠️ **Reinicie o runtime** antes de executar esta seção: `Runtime → Restart runtime`

Utilizamos o subconjunto **Preference** do SmolTalk2 (`llama_3.1_tulu_3_8b_preference_mixture_no_think`), amostrando ~500 exemplos para viabilizar o treinamento no T4.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BÔNUS: DPO — Executar após reiniciar o runtime
# ═══════════════════════════════════════════════════════════════════════

import os, gc, torch
from itertools import islice
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from peft import LoraConfig, PeftModel
from trl import DPOTrainer, DPOConfig

# Login HuggingFace
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    login()

os.environ["WANDB_DISABLED"] = "true"

MODEL_ID = "google/gemma-3-1b-pt"
TOKENIZER_ID = "google/gemma-3-1b-it"
SFT_DIR = "./outputs-sft"
DPO_DIR = "./outputs-dpo"

# ── Carregar dataset de preferências (streaming + amostragem) ──
print("Carregando dataset de preferências...")
ds_pref_stream = load_dataset(
    "HuggingFaceTB/smoltalk2",
    "Preference",
    split="llama_3.1_tulu_3_8b_preference_mixture_no_think",
    streaming=True,
)

DPO_SAMPLES = 500
pref_examples = list(islice(ds_pref_stream, DPO_SAMPLES))
dataset_pref = Dataset.from_list(pref_examples)
print(f"Exemplos de preferência: {len(dataset_pref)}")

# Inspecionar formato
ex = dataset_pref[0]
print(f"\nColunas: {dataset_pref.column_names}")
print(f"Exemplo (chaves): {list(ex.keys())}")

# ── Quantização ──
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# ── Carregar modelo SFT (base + adapters LoRA) ──
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)

model_sft = PeftModel.from_pretrained(base_model, SFT_DIR)
model_sft = model_sft.merge_and_unload()  # Merge para poder adicionar novos adapters

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Modelo SFT carregado e merged")
print(f"   VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Configuração LoRA para DPO ──
lora_config_dpo = LoraConfig(
    r=8,                             # Rank menor (ajuste sutil)
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

# ── Configuração DPO ──
dpo_config = DPOConfig(
    output_dir=DPO_DIR,
    beta=0.1,                         # Controle de desvio da referência
    max_length=512,
    max_prompt_length=256,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=5e-5,               # LR menor para DPO
    warmup_steps=50,
    bf16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=1,
    save_only_model=True,
)

# ── Treinar DPO ──
dpo_trainer = DPOTrainer(
    model=model_sft,
    args=dpo_config,
    train_dataset=dataset_pref,
    processing_class=tokenizer,
    peft_config=lora_config_dpo,
)

print("Iniciando treinamento DPO...")
dpo_trainer.train()

# Salvar
dpo_trainer.save_model(DPO_DIR)
tokenizer.save_pretrained(DPO_DIR)
print(f"\n✅ DPO concluído! Modelo salvo em: {DPO_DIR}")

del dpo_trainer
gc.collect()
torch.cuda.empty_cache()

### Step 13 — Comparação Final: Base vs SFT vs DPO

Comparamos as respostas dos três estágios do pipeline:

1. **Modelo Base** (PT): Geração livre, sem instruções
2. **Modelo SFT**: Segue instruções após fine-tuning supervisionado
3. **Modelo DPO**: Respostas alinhadas com preferências humanas após DPO

A evolução esperada é: base (texto solto) → SFT (segue instruções) → DPO (respostas mais úteis, seguras e alinhadas).

In [ ]:
# ── Carregar modelo DPO para comparação ──
base_model_dpo = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)
model_dpo = PeftModel.from_pretrained(base_model_dpo, DPO_DIR)
model_dpo.config.use_cache = True
model_dpo.gradient_checkpointing_disable()

pipe_dpo = pipeline("text-generation", model=model_dpo, tokenizer=tokenizer)

# Carregar respostas baseline salvas
import json
with open("baseline_responses.json", "r", encoding="utf-8") as f:
    baseline_data = json.load(f)


def gen(pipe, prompt, max_new_tokens=300):
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    stop_ids = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<end_of_turn>"),
    ]
    out = pipe(
        formatted, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.3, top_p=0.9,
        return_full_text=False, eos_token_id=stop_ids,
    )
    return out[0]["generated_text"].strip()


# ── Comparação dos 3 modelos ──
print("=" * 80)
print("  COMPARAÇÃO FINAL: BASE vs SFT vs DPO")
print("=" * 80)

test_prompts = baseline_data["prompts"]

for i, prompt in enumerate(test_prompts):
    print(f"\n{'━' * 80}")
    print(f"📝 PROMPT {i+1}: {prompt}")
    print(f"{'━' * 80}")

    # Base (salvo anteriormente)
    print(f"\n🔴 Modelo Base (PT):")
    print(f"   {baseline_data['responses'][i][:400]}")

    # DPO
    response_dpo = gen(pipe_dpo, prompt)
    print(f"\n🟣 Modelo DPO:")
    print(f"   {response_dpo[:400]}")

    print()

---

## Conclusão

Neste notebook, implementamos o pipeline completo de pós-treinamento de um LLM:

### Resultados Principais

1. **SFT com QLoRA**: Transformamos o Gemma 3 1B PT (modelo de geração livre) em um assistente instrucional, treinando apenas ~1-5% dos parâmetros totais com adaptadores LoRA de rank 16, em quantização 4-bit NF4.

2. **Filtragem Inteligente**: Reduzimos o dataset de 254k exemplos multilíngues para ~2500 exemplos em português, criando um modelo mais especializado no idioma-alvo e viabilizando o treino no Colab gratuito.

3. **Comparação Qualitativa**: Demonstramos que o modelo base apenas completa texto, enquanto o modelo ajustado segue instruções e responde em formato de chat.

4. **DPO (Bônus)**: Aplicamos alinhamento por preferências para refinar as respostas do modelo SFT, usando otimização direta sem necessidade de modelo de recompensa separado.

### Técnicas Utilizadas

| Técnica | Propósito | Referência Teórica |
|---------|-----------|-------------------|
| **QLoRA** | Redução de memória ~8x | Quantização NF4 + LoRA (Dettmers et al., 2023) |
| **LoRA** | Eficiência paramétrica ~99% | Adaptadores de baixo rank (Hu et al., 2021) |
| **SFT** | Instruction following | Fine-tuning supervisionado com chat template |
| **DPO** | Alinhamento com preferências | Otimização direta de preferências (Rafailov et al., 2023) |
| **Packing** | Eficiência de hardware | Combinação de exemplos curtos em sequências longas |
| **Gradient Checkpointing** | Economia de VRAM | Recomputação de ativações no backward pass |